# Lab Day 10: Qwen3-VL-4B QLoRA — bài làm chạy trên Colab T4

Chạy từ trên xuống. Notebook chưa có kết quả GPU; mọi số đo sẽ sinh từ runtime
của bạn. Chuẩn bị bundle `lab_day10_colab_bundle.zip`, Colab Secrets
`KAGGLE_USERNAME`/`KAGGLE_KEY`, và Drive đủ chỗ lưu checkpoint.
Adapter/projector có thể chiếm nhiều GB khi giữ nhiều lượt/checkpoint.
REPORT là bản nháp cần bạn tự đọc ảnh và bổ sung nhận xét cá nhân.

## 0. Môi trường (Colab T4)
Runtime → Change runtime type → T4 GPU. Cell đầu clone repo cá nhân và nạp mã
từ bundle nếu repo GitHub chưa có thư mục solution. Không cần khóa trong mã.

In [ ]:
import os, sys, json, subprocess, zipfile
from pathlib import Path
from google.colab import files, drive, userdata

ROOT = Path('/content/K4-Track-4-Day10-VisionLanguageModel')
REPO_URL = 'https://github.com/hongcuong26-debug/K4-Track-4-Day10-VisionLanguageModel.git'
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
if not (ROOT / 'solution/run.py').exists():
    print('Tải lên lab_day10_colab_bundle.zip đã được chuẩn bị cùng notebook.')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Cần đúng một bundle zip')
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            target = (ROOT / name).resolve()
            if not target.is_relative_to(ROOT.resolve()):
                raise ValueError('Unsafe archive path')
        archive.extractall(ROOT)
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt', 'pytest', 'requests'], check=True)
subprocess.run(['nvidia-smi', '-L'], check=True)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Chọn T4 GPU trước khi tiếp tục'
print(torch.cuda.get_device_name(0), torch.__version__)
assert torch.cuda.get_device_capability(0) == (7, 5), 'Notebook này dành cho T4'
drive.mount('/content/drive')
OUTPUT_ROOT = '/content/drive/MyDrive/day10-vlm/runs'
STUDENT_NAME = 'Nguyễn Hồng Cường'  # Sửa nếu họ tên khác.
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)

run('-m', 'pytest', '-q')
run('-m', 'pytest', '-q', 'student_tests')

## 1. Dữ liệu: tải và chia split
Audit bắt buộc đúng 1422/276/311 ảnh và không có session trùng giữa split.
Để dùng zip dữ liệu thay Kaggle, thay lệnh bằng `--zip /content/dataset.zip`.

In [ ]:
run('scripts/prepare_data.py', '--kaggle')
run('-m', 'solution.inspect_data', '--processor')
run('-m', 'solution.architecture')
from IPython.display import display
from PIL import Image
for path in sorted(Path('report/figures/ground_truth').glob('*.jpg')):
    display(Image.open(path))

## 2. Grounding JSON và E0 zero-shot
Giữ output thô để phân tích lỗi. E0 chạy cả val/test, dùng scorer nguyên bản.

In [ ]:
from solution.grounding import objects_to_answer, PROMPT
from scripts.labcommon import read_jsonl
sample = read_jsonl('data/splits/train.jsonl')[0]
print(PROMPT)
print(objects_to_answer(sample['objects']))
run('-m', 'solution.run', '--config', 'configs/e0.yaml', '--name', 'e0')
print(Path('report/evidence/e0.json').read_text())
for row in read_jsonl('work/e0_test.jsonl')[:5]:
    print(row)

## 3. Chat template và mask loss
Không dịch labels thủ công; Qwen tự shift causal loss. Processor thật đã được
kiểm tra tại chặng 1: xem token ảnh, patch size, và assistant JSON + im_end.

In [ ]:
print(Path('work/encoding_audit.json').read_text())
print(Path('work/architecture.json').read_text())

## 4. Finetune E1, E2, E3
Mỗi lượt bắt đầu từ model gốc. Batch 1 × grad_accum 8 giữ effective batch 8;
giữ đồng nhất cả E1–E3 và E4. 300 optimizer steps/lượt. Checkpoint mỗi 100
bước ghi trực tiếp vào Drive. Nạp model mới + adapter + projector trước test.
Chạy từng cell; không chạy lại lượt đã hoàn thành (dùng inference cell bên dưới).

In [ ]:
run('-m', 'solution.run', '--config', 'configs/e1.yaml', '--name', 'e1', '--batch-size', '1', '--output-root', OUTPUT_ROOT)

In [ ]:
run('-m', 'solution.run', '--config', 'configs/e2.yaml', '--name', 'e2', '--batch-size', '1', '--output-root', OUTPUT_ROOT)

In [ ]:
run('-m', 'solution.run', '--config', 'configs/e3.yaml', '--name', 'e3', '--batch-size', '1', '--output-root', OUTPUT_ROOT)

Nếu session Colab mất sau khi train xong, mount Drive lại, tải/prep dữ liệu
và chạy cell inference dưới đây. Đổi RUN_NAME sang lượt cần khôi phục.
Adapter checkpoint không có optimizer/RNG để resume đúng vòng train giữa chừng.

In [ ]:
RERUN_INFERENCE = False
RUN_NAME = 'e1'
if RERUN_INFERENCE:
    run('-m', 'solution.run', '--config', f'configs/{RUN_NAME}.yaml', '--name', RUN_NAME,
        '--checkpoint', f'{OUTPUT_ROOT}/{RUN_NAME}/final', '--output-root', OUTPUT_ROOT)

## 5. Đánh giá và đọc CI95
Các JSON dưới đây do score.py sinh, kèm VRAM/thời gian thật. Ngưỡng E1 của
rubric được kiểm riêng vì check_submission.py chưa kiểm các ngưỡng này.
Nếu không đạt, gỡ lỗi trên val rồi chạy lại thí nghiệm có ghi chép; không sửa evidence.

In [ ]:
for name in ('e0', 'e1', 'e2', 'e3'):
    ev = json.loads(Path(f'report/evidence/{name}.json').read_text())
    print(name, ev['micro']['f1'], ev['f1_ci95'], ev['meta'])
e0 = json.loads(Path('report/evidence/e0.json').read_text())
e1 = json.loads(Path('report/evidence/e1.json').read_text())
print('E1 parse >= 0.95:', e1['parse_valid_rate'] >= 0.95)
print('E1 - E0 >= 0.10:', e1['micro']['f1'] - e0['micro']['f1'] >= 0.10)

## 6. E4: chỉ đổi data_size
Chọn nền bằng val. 100% dùng lại checkpoint/prediction tương ứng;
10% dùng seed cố định, cùng 300 bước (nhiều epoch hơn), cùng mọi cấu hình khác.

In [ ]:
run('-m', 'solution.ablation', '--output-root', OUTPUT_ROOT)
print(Path('work/e4_manifest.json').read_text())

## 7. E5 GRPO (tùy chọn)
Reward có test cho hoàn hảo, JSON hỏng, sai lớp, thừa box, cùng rỗng và hai
chiến thuật reward hacking. Chỉ bật nếu còn ngân sách T4. Chưa bảo đảm GRPO
không OOM; không coi mã chưa chạy là bằng chứng bonus.

In [ ]:
RUN_E5 = False
if RUN_E5:
    best = json.loads(Path('work/e4_manifest.json').read_text())['base']
    run('-m', 'solution.grpo', '--checkpoint', f'{OUTPUT_ROOT}/{best}/final',
        '--output-root', OUTPUT_ROOT)

## 8. Báo cáo, kiểm tra và tải bài nộp
Sinh báo cáo nháp từ số thật, minh họa GT xanh/dự đoán đỏ, kiểm chứng sai thang.
Bạn phải xem ảnh và tự hoàn thiện hai lỗi E0, đánh đổi tài nguyên, nguyên nhân
lỗi theo lớp và phần nhìn lại. Sau khi sửa REPORT, test gốc kiểm mẫu `<điền`
có thể fail; không sửa tests để né mâu thuẫn này.

Git trên Colab có thể còn dưới 5 commit. Tải zip về repo máy tính và commit
theo các mốc thực hiện trước khi nộp. Bài chỉ hoàn thành khi checker 0 FAIL
và các yêu cầu rubric đều được đáp ứng.

In [ ]:
run('-m', 'solution.report', '--student-name', STUDENT_NAME, '--output-root', OUTPUT_ROOT)
from IPython.display import Markdown
display(Markdown(Path('REPORT.md').read_text()))
result = subprocess.run([sys.executable, 'scripts/check_submission.py'])
print('Checker exit code:', result.returncode)

In [ ]:
# Backup raw predictions and diagnostic metadata privately to Drive.
import shutil
audit_backup = Path(OUTPUT_ROOT).parent / 'audit'
audit_backup.mkdir(parents=True, exist_ok=True)
for path in Path('work').glob('*.json*'):
    shutil.copy2(path, audit_backup / path.name)
shutil.copytree('work/validation', audit_backup / 'validation', dirs_exist_ok=True)
shutil.copytree('report', audit_backup / 'report', dirs_exist_ok=True)
shutil.copy2('REPORT.md', audit_backup / 'REPORT.md')

# Submission bundle excludes data, work, credentials and all model weights.
bundle = Path('/content/day10_results.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in ('solution', 'student_tests', 'report'):
        for path in Path(folder).rglob('*'):
            if path.is_file() and '__pycache__' not in path.parts:
                archive.write(path, str(path))
    for name in ('REPORT.md', 'CHECKLIST.md', 'BAO_CAO_LOI_VA_LUU_Y.md', 'lab_day10.ipynb'):
        archive.write(name, name)
files.download(str(bundle))